# Module 1 — OCR Văn Bản Chữ Hán (PDF Scan / Ảnh Chụp)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caoban123/Tool-Annotation-Sinonom/blob/main/OCR/notebooks/01_OCR_Han.ipynb)

Notebook này thực hiện nhận dạng chữ Hán phồn thể/cổ từ PDF scan ảnh chụp:
- **Tự động kéo dữ liệu:** Kéo tài liệu từ `https://github.com/caoban123/CV01.git` nếu chạy trên Colab GPU.
- **Tự động nhận diện:** Soi xem PDF đã có sẵn text layer chưa; nếu là PDF Scan thì render ảnh và OCR.
- **Cơ chế Checkpoint:** Tự động lưu kết quả theo từng trang, bảo vệ chống mất mát dữ liệu khi Colab ngắt kết nối.
- **Tự động sinh mã ID:** Gán mã tác phẩm, mã đề tài `HVB_01`, SHA-256 provenance và xuất `han.txt`.

In [ ]:
# 1. Cài đặt môi trường & Nạp thư viện OCR
import os
import sys
import subprocess
from pathlib import Path

# Tự động phát hiện và khởi tạo nếu chạy trên Google Colab
if 'google.colab' in sys.modules:
    colab_repo = Path('/content/Tool-Annotation-Sinonom')
    if not colab_repo.exists():
        print('Đang clone repo Tool-Annotation-Sinonom trên Google Colab...')
        subprocess.run(['git', 'clone', 'https://github.com/caoban123/Tool-Annotation-Sinonom.git', str(colab_repo)], check=True)
    os.chdir(str(colab_repo))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'PyMuPDF', 'pillow', 'pydantic'], check=False)
    # Để chạy nhận diện với GPU nâng cao trên Colab, bạn có thể uncomment dòng sau:
    # subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'paddlepaddle-gpu', 'paddleocr'], check=False)

REPO_ROOT = Path('.').resolve()
while REPO_ROOT.name and not (REPO_ROOT / 'OCR').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from PIL import Image
from Crawl.utils.catalog import get_works_catalog, resolve_work, ensure_data_source
from Crawl.utils.id_generator import generate_project_id, generate_document_id, compute_sha256, build_sidecar_metadata
from OCR.utils.ocr_pipeline import BaseOCREngine, OCRPipeline
from OCR.utils.checkpoints import OCRCheckpointManager
from OCR.utils.export_text import export_ocr_result
from OCR.utils.pdf_detector import inspect_pdf

source_dir = ensure_data_source(base_dir=str(REPO_ROOT / 'data' / 'source_materials'))
print('✓ Nạp thư viện OCR và chuẩn bị dữ liệu thành công!')

In [ ]:
# 2. Lựa chọn tác phẩm chữ Hán & Tự động gán mã
catalog = get_works_catalog()
print('=== DANH MỤC CÁC TÁC PHẨM CÓ BẢN HÁN ===')
for k, v in catalog.items():
    if v['han_relpath']:
        print(f"- {k:15s}: {v['name']}")

# Chọn tác phẩm chữ Hán cần OCR (HVNTDDC, LTHCLC_Q20_25, LTHCLC_Q46_49, VIET_SU_YEU, DVDDTB)
SELECTED_WORK_KEY = 'DVDDTB'
READING_ORDER = 'ltr_horizontal'  # 'ltr_horizontal' hoặc 'rtl_vertical'
USE_GPU = False                   # Bật True nếu chạy trên Colab GPU

work_info = resolve_work(SELECTED_WORK_KEY, base_dir=str(source_dir))
target_pdf = work_info['han_abs_path']
print(f"\nĐang chọn: {work_info['name']} ({work_info['ma_tac_pham']})")
print(f"Đường dẫn PDF: {target_pdf}")

In [ ]:
# 3. Tự động soi xét cấu trúc PDF (Text vs Scan)
if not target_pdf or not Path(target_pdf).exists():
    print(f'[LƯU Ý] Chưa thấy file PDF tại {target_pdf}. Dùng ảnh scan mẫu fixture...')
    target_file = str(REPO_ROOT / 'tests' / 'fixtures' / 'sample_han_scan.png')
    is_scan_image = True
else:
    target_file = target_pdf
    is_scan_image = False
    pdf_info = inspect_pdf(target_file, sample_pages=5)
    print('=== THÔNG SỐ TÀI LIỆU ===')
    print(f"Loại tài liệu:   {pdf_info['pdf_type']}")
    print(f"Tổng số trang:   {pdf_info['total_pages']}")
    print(f"Nhận định:       {pdf_info['reason']}")

output_dir = REPO_ROOT / 'data' / 'ocr_output'
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# 4. Khởi tạo Engine và Thực Thi Nhận Dạng (Kèm Checkpoint)
engine = BaseOCREngine(model_name='han', use_gpu=USE_GPU)
pipeline = OCRPipeline(engine=engine, reading_order=READING_ORDER)

if target_file.endswith('.pdf'):
    from OCR.utils.render_pdf import render_pdf_to_images
    # Render mẫu 3 trang đầu để kiểm chứng nhanh
    images = render_pdf_to_images(target_file, max_pages=3)
else:
    images = [Image.open(target_file)]

source_id = f"{work_info['ma_tac_pham']}_han"
print(f"Bắt đầu nhận dạng {len(images)} trang cho nguồn: {source_id}...")
result = pipeline.process_images(images, source_id=source_id)

print('=== KẾT QUẢ OCR CHỮ HÁN ===')
print(f"Số ký tự nhận dạng: {len(result['full_text'])}")
print('\n--- TRÍCH ĐOẠN VĂN BẢN ---')
print(result['full_text'][:300])

In [ ]:
# 5. Xuất file han.txt và Metadata
out_txt = output_dir / 'han.txt'
out_meta = output_dir / 'han_metadata.json'
export_ocr_result(result, out_txt, out_meta)

print(f'✓ Đã xuất kết quả chữ Hán: {out_txt}')
print(f'✓ Đã xuất metadata:          {out_meta}')
print('\nHoàn tất! Sẵn sàng đưa vào Studio để dóng hàng.')